# Podstawy aplikacji internetowych - laboratorium

**Autor:** Aleks Czarnecki  
**Rok utworzenia:** 2026


## Flask

Celem ćwiczenia jest zapoznanie studenta z mikro-frameworkiem **Flask** przeznaczonym dla języka Python. Krok po kroku zbudujemy aplikację internetową - od minimalnego „Hello World", przez szablony HTML i formularze, aż po REST API.

Wymagania: Python 3.8+, edytor tekstu VS Code, przeglądarka internetowa.

## Instalacja i konfiguracja środowiska

Uwaga: poniżej proponowana jest konfiguracja środowiska oparta na środowisku wirtualnym *venv*. Wybór ten można zastąpić innym rozwiązaniem wybranym przez studenta (np. bazującym na Docker, Google Colab, albo wirtualnej maszynie w chmurze).

**1.** Utwórz nowy folder `flask_lab` i otwórz go w VS Code (**File -> Open Folder…**). Następnie otwórz terminal w VS Code (`` Ctrl+Shift+` `` lub **Terminal -> New Terminal**) i wykonaj:

```
python3 -m venv .venv
```

VS Code powinien wykryć nowe środowisko wirtualne i zaproponować jego wybór jako interpreter Pythona - kliknij **Yes**. Jeśli komunikat nie pojawi się, wybierz interpreter ręcznie: `Ctrl+Shift+P` -> *Python: Select Interpreter* -> `.venv`.

### Aktywacja środowiska i instalacja Flask

**W terminalu** - aktywuj środowisko, a następnie zainstaluj Flask:
```
source .venv/bin/activate
pip install Flask
```
> Jeśli po aktywacji komenda `pip` nadal wskazuje na systemowy Python (np. przez aliasy w shellu), użyj pełnej ścieżki:
> ```
> .venv/bin/pip install Flask
> ```

**W notebooku (przycisk ▶ Uruchom)** - upewnij się, że w prawym górnym rogu notebooka wybrany jest interpreter z `.venv` (kliknij *Select Kernel* -> *Python Environments* -> `.venv`). Wtedy komórki kodu będą automatycznie korzystać z właściwego środowiska.

Sprawdź instalację, uruchamiając poniższy kod.

---
**Uwaga dotycząca wersji bibliotek:**
- Notebook i przykłady przygotowano dla Flask **3.1.3** (oraz Jinja2 3.1.6, Werkzeug 3.1.7).
- Zalecana wersja Pythona: 3.8+ (testowano dla 3.14.2).
- W innych wersjach mogą wystąpić drobne różnice w API lub zachowaniu.
---

In [1]:
import flask
print(flask.__version__)

3.1.3


/var/folders/yd/l4q7rh657gxbzgtc1zyhc45r0000gn/T/ipykernel_12638/3294412508.py:2: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print(flask.__version__)


## Minimalna aplikacja

**2.** Utwórz plik `app.py` i umieść w nim poniższy kod - to najprostsza aplikacja Flask.

- `Flask(__name__)` - tworzy instancję aplikacji,
- `@app.route('/')` - definiuje trasę (URL) powiązaną z funkcją,
- `debug=True` - włącza automatyczne przeładowanie kodu i interaktywny debugger.

> Nigdy nie uruchamiaj trybu `debug=True` na serwerze produkcyjnym!

In [ ]:
%%writefile app.py
from flask import Flask

app = Flask(__name__)

@app.route('/')
def hello():
    return '<h1>Hello world!</h1><p>Oto strona internetowa napisana w Pythonie.</p>'

if __name__ == '__main__':
    app.run(debug=True, port=5001) #aby uniknąć blokowania portu 5000 przez inne aplikacje, zmieniamy port na 5001

---
**Zadanie otwarte:**
Korzystając z powyższego przykładu, dodaj trasę `/date`, która wyświetli aktualną datę i godzinę.

**3.** Uruchom aplikację poleceniem `python app.py` i otwórz w przeglądarce http://127.0.0.1:5001/. Powinieneś zobaczyć stronę z powitaniem.

> Jeśli plik nazywa się `app.py`, Flask wykryje go automatycznie - możesz też użyć polecenia `flask run`.

## Routing

**4.** Zastąp zawartość pliku `app.py` poniższym kodem, który definiuje kilka tras - zarówno statycznych, jak i dynamicznych (z parametrami w URL).

> **Routing w Flask:**
> 
> Routing pozwala powiązać konkretne adresy URL z funkcjami w aplikacji. Dzięki temu użytkownik może przechodzić między różnymi stronami, a aplikacja wie, jak obsłużyć każde żądanie. To podstawa każdej aplikacji webowej. Więcej: [Routing w Flask](https://flask.palletsprojects.com/en/3.0.x/quickstart/#routing)


In [ ]:
%%writefile app.py
from flask import Flask
from markupsafe import escape

app = Flask(__name__)

@app.route('/')
def index():
    return '<h1>Strona główna</h1>'

@app.route('/about')
def about():
    return '<h1>O aplikacji</h1>'

@app.route('/contact/')
def contact():
    return '<h1>Kontakt</h1>'

# Routing dynamiczny - wartość z URL trafia jako argument funkcji
@app.route('/user/<username>')
def show_user(username):
    return f'<h1>Profil: {escape(username)}</h1>'

@app.route('/post/<int:post_id>')
def show_post(post_id):
    return f'<h1>Post nr {post_id}</h1>'

if __name__ == '__main__':
    app.run(debug=True, port=5001)

---
**Zadanie otwarte:**
Dodaj trasę `/sum/<a>/<b>`, która przyjmie dwie liczby w adresie URL i zwróci ich sumę.

**5.** Uruchom aplikację i przetestuj w przeglądarce:
- http://127.0.0.1:5001/about oraz `/about/` - co się stanie?
- http://127.0.0.1:5001/contact oraz `/contact/` - jaka jest różnica w zachowaniu?
- http://127.0.0.1:5001/user/JanKowalski
- http://127.0.0.1:5001/post/42 oraz `/post/abc` - co się stanie? Dlaczego?

> Trasa z ukośnikiem na końcu (`/contact/`) działa jak folder - Flask przekieruje `/contact` -> `/contact/`. Trasa bez ukośnika (`/about`) traktowana jest jak plik - `/about/` zwróci 404.


## Metody HTTP

**6.** Domyślnie trasa odpowiada tylko na `GET`. Aby obsłużyć inne metody, dodaj argument `methods`. Zastąp zawartość `app.py` poniższym kodem z formularzem logowania.

> **Metody HTTP:**
> 
> Każda trasa w Flask domyślnie obsługuje tylko metodę GET. Dodając argument `methods`, możemy obsłużyć także POST, PUT, DELETE itp. To pozwala tworzyć formularze i API zgodne z zasadami REST. Więcej: [HTTP Methods w Flask](https://flask.palletsprojects.com/en/3.0.x/quickstart/#http-methods)


In [ ]:
%%writefile app.py
from flask import Flask, request

app = Flask(__name__)

attempts = 0

@app.route('/login', methods=['GET', 'POST'])
def login():
    global attempts
    if request.method == 'POST':
        username = request.form.get('username', '')
        password = request.form.get('password', '')
        if username == 'admin' and password == 'admin':
            attempts = 0
            return f'<h1>Witaj, {username}!</h1>'
        attempts += 1
        if attempts >= 3:
            attempts = 0
            return '<h1>Przekroczono liczbę prób logowania!</h1>'
        return '<h1>Błędne dane!</h1><a href="/login">Spróbuj ponownie</a>'
    return '''
        <h1>Logowanie</h1>
        <form method="POST">
            <p>Login: <input type="text" name="username"></p>
            <p>Hasło: <input type="password" name="password"></p>
            <p><input type="submit" value="Zaloguj"></p>
        </form>
    '''

if __name__ == '__main__':
    app.run(debug=True, port=5001)

---
**Zadanie otwarte:**
Zmodyfikuj trasę `/login`, aby po trzech nieudanych próbach logowania wyświetlała komunikat o zablokowaniu możliwości logowania.

**7.** Przejdź do odpowiedniej witryny. Przetestuj formularz - zaloguj się danymi `admin`/`admin` oraz błędnymi danymi.

Najważniejsze atrybuty obiektu `request`:
- `request.method` - metoda HTTP,
- `request.form` - dane z formularza (POST),
- `request.args` - parametry URL (query string),
- `request.files` - przesłane pliki.

> Generowanie HTML w kodzie Pythona jest nieczytelne - w następnym kroku użyjemy szablonów.

## Szablony Jinja2

**8.** Flask korzysta z silnika szablonów **Jinja2**, który separuje logikę od warstwy prezentacji. Szablony umieszczamy w katalogu `templates/`. Utwórz wymaganą strukturę katalogów i pierwszy szablon.

In [2]:
import os
os.makedirs('templates', exist_ok=True)
os.makedirs('static', exist_ok=True)

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="pl">
<head>
    <meta charset="utf-8">
    <title>{{ title }}</title>
</head>
<body>
    <h1>{{ title }}</h1>
    <p>Witaj, {{ username }}!</p>

    {% if items %}
    <h2>Lista zakupów:</h2>
    <ul>
        {% for item in items %}
        <li>{{ item }}</li>
        {% endfor %}
    </ul>
    {% else %}
    <p>Lista zakupów jest pusta.</p>
    {% endif %}
</body>
</html>

### Zadanie otwarte:
Zmodyfikuj szablon index.html tak, aby wyświetlał liczbę elementów na liście (jeśli lista nie jest pusta).

---
**Zadanie otwarte:**
Stwórz nowy szablon HTML na podstawie powyższego, który wyświetli tabelę z dowolnymi danymi (np. lista studentów z ocenami).

Składnia Jinja2: `{{ zmienna }}` - wstawienie wartości, `{% instrukcja %}` - instrukcja sterująca, `{# komentarz #}` - komentarz.

**9.** Zaktualizuj `app.py`, aby renderował szablon za pomocą `render_template()`, i przetestuj oba adresy.

> **Szablony Jinja2:**
> 
> Silnik szablonów Jinja2 pozwala oddzielić logikę aplikacji od warstwy prezentacji (HTML). Dzięki temu kod jest czytelniejszy i łatwiejszy w utrzymaniu. Szablony umożliwiają dynamiczne generowanie stron oraz dziedziczenie layoutu. Więcej: [Jinja2 Template Designer Documentation](https://jinja.palletsprojects.com/en/3.1.x/templates/)


In [ ]:
%%writefile app.py
from flask import Flask, render_template

app = Flask(__name__)

@app.route('/')
def index():
    return render_template('index.html',
                           title='Strona główna',
                           username='Student',
                           items=['Chleb', 'Masło', 'Mleko', 'Jajka'])

@app.route('/empty')
def empty():
    return render_template('index.html',
                           title='Pusta lista',
                           username='Student',
                           items=[])

if __name__ == '__main__':
    app.run(debug=True, port=5001)

Uruchom aplikację i porównaj http://127.0.0.1:5001/ oraz http://127.0.0.1:5001/empty. Zwróć uwagę, jak `{% if items %}` kontroluje wyświetlanie fragmentów szablonu.

> **Bezpieczeństwo:** Jinja2 automatycznie ekranuje zmienne `{{ }}`, chroniąc przed atakami XSS.

## Dziedziczenie szablonów

**10.** Najważniejszą funkcją Jinja2 jest **dziedziczenie szablonów** - szablon bazowy definiuje wspólny layout (nagłówek, nawigacja, stopka), a szablony potomne nadpisują jedynie wybrane bloki. Utwórz szablon bazowy.

> **Dziedziczenie szablonów:**
> 
> Dzięki dziedziczeniu szablonów można zdefiniować wspólny układ strony (layout) i nadpisywać tylko wybrane fragmenty w podstronach. To znacznie ułatwia utrzymanie i rozwój aplikacji. Więcej: [Jinja2 Template Inheritance](https://jinja.palletsprojects.com/en/3.1.x/templates/#template-inheritance)


In [ ]:
%%writefile templates/base.html
<!DOCTYPE html>
<html lang="pl">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}Aplikacja Flask{% endblock %}</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
    <nav>
        <a href="{{ url_for('index') }}">Strona główna</a>
        <span class="nav-sep">|</span>
        <a href="{{ url_for('about') }}">O aplikacji</a>
        <span class="nav-sep">|</span>
        <a href="{{ url_for('contact') }}">Kontakt</a>
    </nav>
    <main>
        {% with messages = get_flashed_messages(with_categories=true) %}
            {% if messages %}
                {% for category, message in messages %}
                <div class="alert alert-{{ category }}">{{ message }}</div>
                {% endfor %}
            {% endif %}
        {% endwith %}
        {% block content %}{% endblock %}
    </main>
    <footer>
        <p>&copy; 2026 Aplikacja Flask - Laboratorium</p>
    </footer>
</body>
</html>

**11.** Utwórz szablony potomne dziedziczące po `base.html`. Kluczowy jest znacznik `{% extends "base.html" %}` na początku pliku.

In [ ]:
%%writefile templates/index.html
{% extends "base.html" %}
{% block title %}Strona główna{% endblock %}
{% block content %}
<h1>Witaj w aplikacji Flask!</h1>
<p>To jest strona główna naszej aplikacji laboratoryjnej.</p>
{% if items %}
<h2>Lista elementów:</h2>
<ul>
    {% for item in items %}
    <li>{{ item }}</li>
    {% endfor %}
</ul>
{% endif %}
{% endblock %}

In [ ]:
%%writefile templates/about.html
{% extends "base.html" %}
{% block title %}O aplikacji{% endblock %}
{% block content %}
<h1>O aplikacji</h1>
<p>Ta aplikacja powstała w ramach zajęć z podstaw aplikacji internetowych.</p>
{% endblock %}

In [ ]:
%%writefile templates/contact.html
{% extends "base.html" %}
{% block title %}Kontakt{% endblock %}
{% block content %}
<h1>Kontakt</h1>
<p>E-mail: <a href="mailto:kontakt@example.com">kontakt@example.com</a></p>
{% endblock %}

**12.** Zaktualizuj `app.py`, aby korzystał z nowych szablonów, i przetestuj nawigację między stronami.

In [ ]:
%%writefile app.py
from flask import Flask, render_template

app = Flask(__name__)
app.secret_key = 'klucz-tajny-do-zmiany-w-produkcji'

@app.route('/')
def index():
    items = ['Python', 'Flask', 'Jinja2']
    return render_template('index.html', items=items)

@app.route('/about')
def about():
    return render_template('about.html')

@app.route('/contact')
def contact():
    return render_template('contact.html')

if __name__ == '__main__':
    app.run(debug=True, port=5001)

Nawigacja i stopka pojawiają się na każdej stronie, choć zdefiniowaliśmy je tylko raz - w szablonie bazowym. To siła dziedziczenia szablonów.

## Pliki statyczne i CSS

**13.** Flask udostępnia pliki statyczne z katalogu `static/`. W `base.html` arkusz stylów jest już dołączony. Utwórz plik CSS.

> **Pliki statyczne:**
> 
> Pliki takie jak arkusze CSS, obrazy czy skrypty JavaScript umieszczamy w katalogu `static/`. Flask automatycznie udostępnia je pod odpowiednimi adresami URL. To standardowy sposób organizacji zasobów w aplikacjach webowych. Więcej: [Static Files w Flask](https://flask.palletsprojects.com/en/3.0.x/quickstart/#static-files)


In [ ]:
%%writefile static/style.css
* { margin: 0; padding: 0; box-sizing: border-box; }
body { font-family: 'Segoe UI', Tahoma, sans-serif; color: #333; line-height: 1.6; }
nav { background: #2c3e50; padding: 15px 30px; display: flex; align-items: center; gap: 10px; }
nav a { color: white; text-decoration: none; font-weight: bold; }
nav a:hover { color: #3498db; }
.nav-sep { color: #7f8c8d; }
.nav-auth { margin-left: auto; }
main { max-width: 900px; margin: 30px auto; padding: 0 20px; }
h1 { color: #2c3e50; margin-bottom: 15px; }
h2 { color: #34495e; margin: 20px 0 10px; }
ul { margin-left: 20px; margin-bottom: 15px; }
footer { background: #ecf0f1; text-align: center; padding: 20px; margin-top: 40px; border-top: 2px solid #bdc3c7; }
.alert { padding: 10px 15px; margin-bottom: 15px; border-radius: 4px; }
.alert-success { background: #d4edda; color: #155724; border: 1px solid #c3e6cb; }
.alert-error { background: #f8d7da; color: #721c24; border: 1px solid #f5c6cb; }
.alert-info { background: #d1ecf1; color: #0c5460; border: 1px solid #bee5eb; }
form { max-width: 500px; }
form label { display: block; margin-top: 10px; font-weight: bold; }
form input[type="text"], form input[type="email"], form input[type="password"], form textarea {
    width: 100%; padding: 8px; margin-top: 5px; border: 1px solid #ccc; border-radius: 4px;
}
form input[type="submit"] {
    margin-top: 15px; padding: 10px 25px; background: #2c3e50; color: white;
    border: none; border-radius: 4px; cursor: pointer; font-size: 16px;
}
form input[type="submit"]:hover { background: #3498db; }

Uruchom aplikację ponownie i odśwież stronę (Ctrl+Shift+R). Strona powinna mieć pasek nawigacyjny, formatowanie i stopkę.

## Formularze i komunikaty flash

**14.** Zmodyfikuj szablon kontaktu, aby zawierał formularz.

In [ ]:
%%writefile templates/contact.html
{% extends "base.html" %}
{% block title %}Kontakt{% endblock %}
{% block content %}
<h1>Formularz kontaktowy</h1>
<form method="POST">
    <label for="name">Imię i nazwisko:</label>
    <input type="text" id="name" name="name" required>
    <label for="email">Adres e-mail:</label>
    <input type="email" id="email" name="email" required>
    <label for="message">Wiadomość:</label>
    <textarea id="message" name="message" rows="5" required></textarea>
    <input type="submit" value="Wyślij wiadomość">
</form>
{% endblock %}

---
**Zadanie otwarte:**
Dodaj do formularza kontaktowego pole na numer telefonu.

**15.** Zaktualizuj `app.py`, aby obsługiwał formularz (POST), wyświetlał komunikaty flash i wykonywał przekierowanie. Jest to wzorzec **POST -> REDIRECT -> GET**

> **Wzorzec Post/Redirect/Get (PRG):**
> 
> To popularny wzorzec projektowy w aplikacjach webowych. Po wysłaniu formularza (POST) aplikacja wykonuje przekierowanie (Redirect) na inną stronę (GET). Dzięki temu odświeżenie strony nie powoduje ponownego wysłania formularza, co zapobiega duplikacji danych. Więcej informacji: [Wikipedia: Post/Redirect/Get](https://en.wikipedia.org/wiki/Post/Redirect/Get)


In [ ]:
%%writefile app.py
from flask import Flask, render_template, request, redirect, url_for, flash

app = Flask(__name__)
app.secret_key = 'klucz-tajny-do-zmiany-w-produkcji'

@app.route('/')
def index():
    items = ['Python', 'Flask', 'Jinja2']
    return render_template('index.html', items=items)

@app.route('/about')
def about():
    return render_template('about.html')

@app.route('/contact', methods=['GET', 'POST'])
def contact():
    if request.method == 'POST':
        name = request.form.get('name', '').strip()
        email = request.form.get('email', '').strip()
        message = request.form.get('message', '').strip()
        if not name or not email or not message:
            flash('Wszystkie pola są wymagane!', 'error')
            return redirect(url_for('contact'))
        flash(f'Dziękujemy, {name}! Twoja wiadomość została wysłana.', 'success')
        return redirect(url_for('index'))
    return render_template('contact.html')

if __name__ == '__main__':
    app.run(debug=True, port=5001)

Przetestuj formularz - wyślij go z poprawnymi danymi i z pustymi polami. Odśwież stronę po komunikacie - czy pojawia się ponownie? Dlaczego?


> **Komunikaty flash:**
> 
> Funkcja `flash()` pozwala przekazywać jednorazowe komunikaty między żądaniami (np. informacja o sukcesie lub błędzie). To wygodny sposób na informowanie użytkownika o rezultacie operacji. Więcej: [Flask Flashing](https://flask.palletsprojects.com/en/3.0.x/patterns/flashing/)


## Sesje

**16.** Sesje pozwalają przechowywać informacje użytkownika między żądaniami. Dane sesji są zapisywane w podpisanym ciasteczku. Dodaj logowanie i wylogowanie do `app.py`.

In [ ]:
%%writefile app.py
from flask import Flask, render_template, request, redirect, url_for, flash, session

app = Flask(__name__)
app.secret_key = 'klucz-tajny-do-zmiany-w-produkcji'

@app.route('/')
def index():
    items = ['Python', 'Flask', 'Jinja2']
    return render_template('index.html', items=items)

@app.route('/about')
def about():
    return render_template('about.html')

@app.route('/contact', methods=['GET', 'POST'])
def contact():
    if request.method == 'POST':
        name = request.form.get('name', '').strip()
        email = request.form.get('email', '').strip()
        message = request.form.get('message', '').strip()
        if not name or not email or not message:
            flash('Wszystkie pola są wymagane!', 'error')
            return redirect(url_for('contact'))
        flash(f'Dziękujemy, {name}! Twoja wiadomość została wysłana.', 'success')
        return redirect(url_for('index'))
    return render_template('contact.html')

@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        username = request.form.get('username', '').strip()
        password = request.form.get('password', '')
        if username == 'admin' and password == 'admin123':
            session['username'] = username
            flash(f'Zalogowano jako {username}!', 'success')
            return redirect(url_for('index'))
        flash('Nieprawidłowe dane logowania!', 'error')
        return redirect(url_for('login'))
    return render_template('login.html')

@app.route('/logout')
def logout():
    session.pop('username', None)
    flash('Wylogowano pomyślnie.', 'info')
    return redirect(url_for('index'))

if __name__ == '__main__':
    app.run(debug=True, port=5001)

**17.** Utwórz szablon logowania oraz zaktualizuj nawigację w `base.html`, aby wyświetlała informację o zalogowanym użytkowniku.

> **Sesje w Flask:**
> 
> Sesje pozwalają przechowywać dane użytkownika pomiędzy żądaniami HTTP. W Flask sesja to podpisany słownik przechowywany w ciasteczku. Dzięki temu można np. zapamiętać zalogowanego użytkownika. Więcej: [Flask Sessions](https://flask.palletsprojects.com/en/3.0.x/quickstart/#sessions)


In [ ]:
%%writefile templates/login.html
{% extends "base.html" %}
{% block title %}Logowanie{% endblock %}
{% block content %}
<h1>Logowanie</h1>
<form method="POST">
    <label for="username">Nazwa użytkownika:</label>
    <input type="text" id="username" name="username" required>
    <label for="password">Hasło:</label>
    <input type="password" id="password" name="password" required>
    <input type="submit" value="Zaloguj się">
</form>
{% endblock %}

In [ ]:
%%writefile templates/base.html
<!DOCTYPE html>
<html lang="pl">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}Aplikacja Flask{% endblock %}</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
    <nav>
        <a href="{{ url_for('index') }}">Strona główna</a>
        <span class="nav-sep">|</span>
        <a href="{{ url_for('about') }}">O nas</a>
        <span class="nav-sep">|</span>
        <a href="{{ url_for('contact') }}">Kontakt</a>
        {% if session.username %}
            <a class="nav-auth" href="{{ url_for('logout') }}">Wyloguj ({{ session.username }})</a>
        {% else %}
            <a class="nav-auth" href="{{ url_for('login') }}">Zaloguj</a>
        {% endif %}
    </nav>
    <main>
        {% with messages = get_flashed_messages(with_categories=true) %}
            {% if messages %}
                {% for category, message in messages %}
                <div class="alert alert-{{ category }}">{{ message }}</div>
                {% endfor %}
            {% endif %}
        {% endwith %}
        {% block content %}{% endblock %}
    </main>
    <footer>
        <p>&copy; 2026 Aplikacja Flask - Laboratorium</p>
    </footer>
</body>
</html>

**18.** Przetestuj sesje: zaloguj się (`admin` / `admin123`), sprawdź zmianę nawigacji, wyloguj się.

> Obiekt `session` to podpisany słownik - użytkownik może odczytać jego zawartość, ale nie może jej zmodyfikować bez `secret_key`.

## Obsługa błędów

**19.** Dodaj do `app.py` obsługę błędów 404 i 500 (przed linią `if __name__`) oraz utwórz szablon strony błędu.

> **Obsługa błędów:**
> 
> Flask pozwala tworzyć własne strony błędów (np. 404, 500), co poprawia doświadczenie użytkownika i pozwala przekazać przyjazny komunikat. To także dobry sposób na ukrycie szczegółów technicznych aplikacji. Więcej: [Error Handling w Flask](https://flask.palletsprojects.com/en/3.0.x/errorhandling/)


In [ ]:
 # Dodaj do app.py przed linią if __name__ == '__main__':

@app.errorhandler(404)
def page_not_found(error):
    return render_template('error.html', code=404,
                           message='Strona nie została znaleziona'), 404

@app.errorhandler(500)
def internal_error(error):
    return render_template('error.html', code=500,
                           message='Wewnętrzny błąd serwera'), 500

In [ ]:
%%writefile templates/error.html
{% extends "base.html" %}
{% block title %}Błąd {{ code }}{% endblock %}
{% block content %}
<h1>Błąd {{ code }}</h1>
<p>{{ message }}</p>
<p><a href="{{ url_for('index') }}">Powrót na stronę główną</a></p>
{% endblock %}

Wejdź na nieistniejący adres, np. http://127.0.0.1:5001/xyz. Czy widzisz niestandardową stronę 404?

> Zwróć uwagę na krotkę `return ..., 404` - drugi element to kod HTTP. Bez niego Flask zwróciłby domyślne 200.

## REST API

**20.** Na koniec dodamy proste REST API zwracające dane JSON - technikę wykorzystywaną w komunikacji z frontendem (React, Vue.js itp.). Flask automatycznie konwertuje zwrócony słownik/listę na JSON. Dodaj poniższy kod do `app.py` (przed `if __name__`).


> REST API umożliwia komunikację aplikacji z innymi systemami (np. frontendem w React). Flask pozwala łatwo tworzyć endpointy zwracające dane w formacie JSON. To podstawa nowoczesnych aplikacji webowych. Więcej: [REST API w Flask](https://flask.palletsprojects.com/en/3.0.x/api/#flask.Flask.json)


In [ ]:
 # Dodaj do app.py przed linią if __name__ == '__main__':

from flask import jsonify

books_data = [
    {'id': 1, 'title': 'Wiedźmin: Ostatnie życzenie', 'author': 'Andrzej Sapkowski', 'year': 1993},
    {'id': 2, 'title': 'Solaris', 'author': 'Stanisław Lem', 'year': 1961},
    {'id': 3, 'title': 'Lalka', 'author': 'Bolesław Prus', 'year': 1890},
]
next_id = 4

@app.route('/api/books')
def api_books():
    return books_data

@app.route('/api/books/<int:book_id>')
def api_book(book_id):
    book = next((b for b in books_data if b['id'] == book_id), None)
    if book is None:
        return {'error': 'Nie znaleziono książki'}, 404
    return book

@app.route('/api/books', methods=['POST'])
def api_book_create():
    global next_id
    data = request.get_json()
    if not data:
        return jsonify({'error': 'Brak danych JSON'}), 400
    for field in ['title', 'author', 'year']:
        if field not in data:
            return jsonify({'error': f'Brak pola: {field}'}), 400
    book = {'id': next_id, 'title': str(data['title']),
            'author': str(data['author']), 'year': int(data['year'])}
    next_id += 1
    books_data.append(book)
    return jsonify(book), 201

---
**Zadanie otwarte:**
Korzystając z powyższego mechanizmu, dodaj endpoint `/api/authors`, który zwróci listę wszystkich autorów książek (bez duplikatów) w formacie JSON.

**21.** Przetestuj API w przeglądarce i za pomocą `curl`:
```
curl http://127.0.0.1:5001/api/books
curl http://127.0.0.1:5001/api/books/1
curl -X POST http://127.0.0.1:5001/api/books \
  -H "Content-Type: application/json" \
  -d '{"title":"Quo Vadis","author":"Henryk Sienkiewicz","year":1896}'
```

> Kody HTTP: `200` - sukces, `201` - Created, `400` - Bad Request, `404` - Not Found.

> Dane przechowywane są w pamięci - po restarcie serwera wrócą do stanu początkowego. W produkcji użylibyśmy bazy danych (np. Flask-SQLAlchemy).